In [1]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
MHE + NMPC — IMPLEMENTACIÓN PLANTA FÍSICA REAL (con sensor verdad-terreno h₂)
==============================================================================
Igual que la versión anterior pero agrega:
  - Lectura del tercer canal h₂ del ESP32 (verdad-terreno, solo verificación)
  - H2_REAL[k] almacena la medición real de h₂ en cada paso
  - compute_metrics() calcula RMSE/bias del observador MHE vs. real
  - plot_results() muestra panel comparativo estimado vs. real en h₂
  - CSV incluye columna h2_real_m y error_obs_cm
  - Tabla y resumen reportan métricas del observador (igual que Koopman v4b)

El lazo de control es IDÉNTICO al original — h₂_real NO entra al MHE ni al NMPC.
"""

import numpy as np
from scipy.optimize import least_squares
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import serial
import serial.tools.list_ports
import threading
import queue
import sys
import csv
from datetime import datetime

warnings.filterwarnings('ignore')

output_dir = Path('./outputs_real')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS
# ============================================================================
P_NOM = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,
    'eta'     : 8.9e-4,
    'g'       : 9.81,
    'alphao1' : 0.0271,  'Do1': 10e-3,
    'alphao2' : 0.0271,  'A2' : 7.85e-5,
    'alphao3' : 0.0271,  'Do3': 10e-3,
    'alpha120': 0.3038,  'D12': 10e-3,  'A12': 7.85e-5,  'lambdac12': 24000,
    'alpha230': 0.1344,  'D23': 10e-3,  'A23': 7.85e-5,  'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

# ============================================================================
# PARÁMETROS DE CONTROL / EXPERIMENTO
# ============================================================================
Ts           = 2.0
T_sim        = 800
Nsim         = int(T_sim / Ts)
SETTLING_S   = 60

SERIAL_PORT    = 'COM7'
BAUD_RATE      = 115200
SERIAL_TIMEOUT = 3.0
MAX_DELTA_H    = 0.05   # m — filtro anti-outlier

REF_STEPS = [
    (0,              0.10),
    (int(Nsim*0.33), 0.20),
    (int(Nsim*0.66), 0.15),
]

EPS   = 1e-10
EPS_S = 1e-8

# ============================================================================
# COMUNICACIÓN SERIAL — ahora lee los tres canales (h1, h2_real, h3)
# ============================================================================
class ESP32Interface:
    def __init__(self, port=SERIAL_PORT, baud=BAUD_RATE):
        self.port = port; self.baud = baud; self.ser = None
        self.rx_queue    = queue.Queue(maxsize=20)
        self._stop_event = threading.Event()
        self._last_meas  = None   # (h1, h2_real, h3)
        self._lock       = threading.Lock()
        self.outlier_count = 0

    def connect(self):
        print(f"\nConectando a {self.port} @ {self.baud} baud...")
        try:
            self.ser = serial.Serial(self.port, self.baud,
                                     timeout=SERIAL_TIMEOUT, write_timeout=1.0)
            time.sleep(2.0)
            self.ser.reset_input_buffer()
            print(f"  ✓ Conectado a {self.port}")
        except serial.SerialException as e:
            print(f"  ✗ Error: {e}")
            for p in serial.tools.list_ports.comports():
                print(f"    {p.device} — {p.description}")
            raise
        self._reader_thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._reader_thread.start()

    def _reader_loop(self):
        while not self._stop_event.is_set():
            try:
                if self.ser.in_waiting:
                    line = self.ser.readline().decode('utf-8', errors='ignore').strip()
                    if line:
                        self.rx_queue.put_nowait(line)
            except Exception:
                pass
            time.sleep(0.005)

    def get_measurement(self, timeout=3.0):
        """
        Devuelve (h1, h2_real, h3) en metros.

        h2_real es el sensor de verdad-terreno — NO entra al MHE ni al NMPC.
        Solo se almacena para comparar con la estimación del MHE.
        """
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                line = self.rx_queue.get(timeout=0.1)
                line = line.replace('\x00', '').replace('\0', '').strip()
                line = ''.join(c for c in line if c.isprintable())
                if not line: continue
                parts = line.split(',')
                if len(parts) != 3: continue
                try:
                    h1      = max(float(parts[0].strip()), 0.0)
                    h2_real = max(float(parts[1].strip()), 0.0)
                    h3      = max(float(parts[2].strip()), 0.0)
                except (ValueError, OverflowError):
                    continue

                # Filtro anti-outlier en los tres canales
                with self._lock:
                    if self._last_meas is not None:
                        h1p, h2p, h3p = self._last_meas
                        if abs(h1 - h1p) > MAX_DELTA_H:
                            h1 = h1p; self.outlier_count += 1
                        if abs(h2_real - h2p) > MAX_DELTA_H:
                            h2_real = h2p; self.outlier_count += 1
                        if abs(h3 - h3p) > MAX_DELTA_H:
                            h3 = h3p; self.outlier_count += 1
                    self._last_meas = (h1, h2_real, h3)

                return h1, h2_real, h3

            except queue.Empty:
                pass

        with self._lock:
            if self._last_meas is not None:
                print("  ⚠ Timeout serial — usando última medición")
                return self._last_meas
        raise TimeoutError("No se recibieron datos del ESP32")

    def send_control(self, u1: float, u2: float):
        cmd = f"{float(np.clip(u1,0,1)):.4f},{float(np.clip(u2,0,1)):.4f}\n"
        try:
            self.ser.write(cmd.encode('utf-8'))
        except Exception as e:
            print(f"  ✗ Error al enviar: {e}")

    def stop_pumps(self):
        self.send_control(0.0, 0.0)
        print("  Bombas apagadas.")

    def close(self):
        self._stop_event.set()
        self.stop_pumps()
        time.sleep(0.5)
        if self.ser and self.ser.is_open:
            self.ser.close()
        print("  Puerto serial cerrado.")

# ============================================================================
# MODELO NO LINEAL (Python)
# ============================================================================
def nl_ode_py(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh12)+EPS)
    a12  = p['alpha120']*np.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12']*np.sqrt(2*p['g']*abs(dh12)+EPS)*np.sign(dh12+1e-15)
    dh23 = h2 - h3
    l23  = p['D23']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh23)+EPS)
    a23  = p['alpha230']*np.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23']*np.sqrt(2*p['g']*abs(dh23)+EPS)*np.sign(dh23+1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4_py(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode_py(x, u, p); k2 = nl_ode_py(x+dt/2*k1, u, p)
    k3 = nl_ode_py(x+dt/2*k2, u, p); k4 = nl_ode_py(x+dt*k3, u, p)
    return np.clip(x + dt/6*(k1+2*k2+2*k3+k4), 0, 0.22)

def compute_ss(h2_ref, p=P_NOM):
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode_py([max(h1,1e-4), h2_ref, max(h3,1e-4)],
                         [np.clip(u1,0,1), 0.0], p)
    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5), (h2_ref*1.6, h2_ref*0.7, 0.7),
        (h2_ref*2.0, h2_ref*0.8, 0.9), (h2_ref*2.5, h2_ref*0.8, 0.7),
        (h2_ref*2.5, h2_ref*0.8, 0.75), (h2_ref*1.8, h2_ref*0.85, 0.85),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01,0.01,0.0],[0.22,0.22,1.0]),
                              max_nfev=5000)
            if s.success and np.linalg.norm(s.fun) < best_res:
                best_res = np.linalg.norm(s.fun); best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2],0,1), 0.0])
    return None, None

# ============================================================================
# CONSTRUCCIÓN DE MHE Y NMPC CON CASADI
# ============================================================================
def build_solvers(p=P_NOM):
    print("\nConstruyendo modelo CasADi...")
    nx, nu, ny = 3, 2, 2

    x_s = ca.SX.sym('x', nx); u_s = ca.SX.sym('u', nu)
    h1, h2, h3 = x_s[0], x_s[1], x_s[2]

    qi1 = p['qi1max'] * u_s[0]
    qi3 = p['qi3max'] * u_s[1]

    h1p = (h1 + ca.sqrt(h1**2 + EPS_S)) / 2
    h2p = (h2 + ca.sqrt(h2**2 + EPS_S)) / 2
    h3p = (h3 + ca.sqrt(h3**2 + EPS_S)) / 2

    qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*ca.sqrt(2*p['g']*h1p)
    qo2 = p['alphao2']*p['A2']              *ca.sqrt(2*p['g']*h2p)
    qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*ca.sqrt(2*p['g']*h3p)

    dh12 = h1-h2; abs12 = ca.sqrt(dh12**2+EPS_S)
    l12  = p['D12']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs12)
    a12  = p['alpha120']*ca.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12']*ca.sqrt(2*p['g']*abs12)*ca.sign(dh12)

    dh23 = h2-h3; abs23 = ca.sqrt(dh23**2+EPS_S)
    l23  = p['D23']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs23)
    a23  = p['alpha230']*ca.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23']*ca.sqrt(2*p['g']*abs23)*ca.sign(dh23)

    xdot = ca.vertcat(
        (qi1-q12-qo1)/p['Atank'],
        (q12-q23-qo2)/p['Atank'],
        (qi3+q23-qo3)/p['Atank'],
    )
    f_cas = ca.Function('f', [x_s, u_s], [xdot])
    k1r = f_cas(x_s, u_s); k2r = f_cas(x_s+Ts/2*k1r, u_s)
    k3r = f_cas(x_s+Ts/2*k2r, u_s); k4r = f_cas(x_s+Ts*k3r, u_s)
    F_rk4 = ca.Function('F', [x_s, u_s], [x_s + Ts/6*(k1r+2*k2r+2*k3r+k4r)])
    C_mat = np.array([[1,0,0],[0,0,1]], dtype=float)   # mide h1, h3
    print("  Modelo RK4 listo.")

    # ── MHE ──────────────────────────────────────────────────────────────────
    print("  Construyendo MHE...")
    N_mhe = 20
    S_mhe = np.diag([50.0, 0.5, 50.0])
    Q_mhe = np.diag([10.0, 50.0, 10.0])
    R_mhe = np.diag([1.0, 1.0])
    xmin_mhe = np.array([0.0, 0.0, 0.0])
    xmax_mhe = np.array([0.22, 0.22, 0.22])

    X_mhe  = ca.SX.sym('X', nx, N_mhe+1)
    xbar_p = ca.SX.sym('xbar', nx)
    Uk_p   = ca.SX.sym('Uk', nu, N_mhe)
    Yk_p   = ca.SX.sym('Yk', ny, N_mhe)

    J_mhe = 0
    e0 = X_mhe[:,0] - xbar_p
    J_mhe += ca.mtimes(e0.T, ca.DM(S_mhe) @ e0)

    for j in range(N_mhe):
        xj  = X_mhe[:, j]; xj1 = X_mhe[:, j+1]
        uj  = Uk_p[:, j];  yj  = Yk_p[:, j]
        wj  = xj1 - F_rk4(xj, uj)
        vj  = yj - ca.DM(C_mat) @ xj
        J_mhe += ca.mtimes(wj.T, ca.DM(Q_mhe) @ wj)
        J_mhe += ca.mtimes(vj.T, ca.DM(R_mhe) @ vj)

    w_mhe   = ca.reshape(X_mhe, nx*(N_mhe+1), 1)
    par_mhe = ca.vertcat(xbar_p,
                         ca.reshape(Uk_p, nu*N_mhe, 1),
                         ca.reshape(Yk_p, ny*N_mhe, 1))
    lbw_mhe = np.tile(xmin_mhe, N_mhe+1)
    ubw_mhe = np.tile(xmax_mhe, N_mhe+1)

    nlp_mhe  = {'x': w_mhe, 'f': J_mhe, 'p': par_mhe}
    opts_mhe = {'ipopt.print_level': 0, 'ipopt.max_iter': 200,
                'ipopt.tol': 1e-4, 'print_time': 0}
    solver_mhe = ca.nlpsol('solver_mhe', 'ipopt', nlp_mhe, opts_mhe)
    print(f"  MHE listo (N={N_mhe}).")

    # ── NMPC ─────────────────────────────────────────────────────────────────
    print("  Construyendo NMPC...")
    N_mpc = 20; q_mpc = 500
    R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])
    xmin_mpc = np.array([0.0, 0.0, 0.0])
    xmax_mpc = np.array([0.22, 0.22, 0.22])
    cT = np.array([0.0, 1.0, 0.0])

    X_d  = ca.SX.sym('X', nx, N_mpc+1); U_d = ca.SX.sym('U', nu, N_mpc)
    x0_p = ca.SX.sym('x0', nx); yr_p = ca.SX.sym('yr'); up_p = ca.SX.sym('up', nu)

    J_mpc = 0; g = []; lbg = []; ubg = []
    g   += [X_d[:,0] - x0_p]; lbg += [0.0]*nx; ubg += [0.0]*nx

    for k in range(N_mpc):
        g   += [X_d[:,k+1] - F_rk4(X_d[:,k], U_d[:,k])]
        lbg += [0.0]*nx; ubg += [0.0]*nx
        yk  = ca.dot(ca.DM(cT), X_d[:,k+1])
        ey  = yk - yr_p
        du  = U_d[:,0]-up_p if k==0 else U_d[:,k]-U_d[:,k-1]
        wk  = 10 if k == N_mpc-1 else 1
        J_mpc += wk*q_mpc*ey**2
        J_mpc += ca.mtimes(U_d[:,k].T, ca.DM(R1) @ U_d[:,k])
        J_mpc += ca.mtimes(du.T, ca.DM(R2) @ du)

    w_mpc   = ca.vertcat(ca.reshape(X_d, nx*(N_mpc+1), 1),
                         ca.reshape(U_d, nu*N_mpc, 1))
    par_mpc = ca.vertcat(x0_p, yr_p, up_p)
    lbw_mpc = np.concatenate([np.tile(xmin_mpc, N_mpc+1), np.tile(umin, N_mpc)])
    ubw_mpc = np.concatenate([np.tile(xmax_mpc, N_mpc+1), np.tile(umax, N_mpc)])

    nlp_mpc  = {'x': w_mpc, 'f': J_mpc, 'g': ca.vertcat(*g), 'p': par_mpc}
    opts_mpc = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
                'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    solver_mpc = ca.nlpsol('solver_mpc', 'ipopt', nlp_mpc, opts_mpc)
    print(f"  NMPC listo (N={N_mpc}, q={q_mpc}).")

    return (solver_mhe, N_mhe, xmin_mhe, xmax_mhe, S_mhe, Q_mhe, R_mhe,
            lbw_mhe, ubw_mhe, C_mat, nx, nu, ny,
            solver_mpc, N_mpc, umin, umax, lbw_mpc, ubw_mpc, lbg, ubg)

# ============================================================================
# LOOP DE CONTROL EN TIEMPO REAL
# ============================================================================
def run_real_experiment(esp, SS_NOM, solvers):
    (solver_mhe, N_mhe, xmin_mhe, xmax_mhe, S_mhe, Q_mhe, R_mhe,
     lbw_mhe, ubw_mhe, C_mat, nx, nu, ny,
     solver_mpc, N_mpc, umin, umax, lbw_mpc, ubw_mpc, lbg, ubg) = solvers

    yref_v = np.zeros(Nsim)
    for (k_start, ref_val) in REF_STEPS:
        yref_v[k_start:] = ref_val
    ref_change_times_s = [k*Ts for k, _ in REF_STEPS[1:]]

    print("\nEsperando primera medición del ESP32...")
    h1_meas, h2_real, h3_meas = esp.get_measurement()
    print(f"  Primera medición: h1={h1_meas*100:.2f}cm | "
          f"h2_real={h2_real*100:.2f}cm | h3={h3_meas*100:.2f}cm")

    x_ss_init, u_ss_init = SS_NOM.get(0.10, (None, None))
    if x_ss_init is None:
        x_ss_init = np.array([h1_meas, yref_v[0], h3_meas])
        u_ss_init = np.array([0.3, 0.0])

    Uk_buf  = np.tile(u_ss_init, (N_mhe, 1)).T
    Yk_buf  = np.tile(C_mat @ x_ss_init, (N_mhe, 1)).T
    xbar    = x_ss_init.copy()
    w0_mhe  = np.tile(x_ss_init, N_mhe+1)

    uprev   = u_ss_init.copy()
    Xw      = np.tile(x_ss_init, (N_mpc+1, 1)).T
    Uw      = np.tile(u_ss_init, (N_mpc, 1)).T
    w0_mpc  = np.concatenate([Xw.flatten(order='F'), Uw.flatten(order='F')])

    # Historial — ahora incluye h2_real
    T_hist     = np.arange(Nsim+1) * Ts
    H_meas     = np.zeros((2, Nsim+1))   # h1, h3 medidos
    H2_REAL    = np.zeros(Nsim+1)        # h2 real — solo verificación
    X_est      = np.zeros((3, Nsim+1))   # estimación MHE [h1, h2, h3]
    U_hist     = np.zeros((2, Nsim))
    t_mhe_ms   = []
    t_mpc_ms   = []
    t_total_ms = []

    H_meas[0, 0]  = h1_meas
    H_meas[1, 0]  = h3_meas
    H2_REAL[0]    = h2_real
    X_est[:, 0]   = x_ss_init

    print(f"\n{'='*72}")
    print(f"  MHE+NMPC PLANTA REAL (con sensor verdad-terreno h₂) — "
          f"{datetime.now().strftime('%H:%M:%S')}")
    print(f"  h₂_real: solo verificación — NO entra al MHE ni al NMPC")
    print(f"{'='*72}")
    print(f"  {'t[s]':>6} | {'h1[cm]':>7} | {'ĥ2[cm]':>7} | {'h2r[cm]':>7} | "
          f"{'eobs[cm]':>8} | {'ref[cm]':>7} | {'u1':>5} | {'MHE':>6} | {'MPC':>6}")
    print(f"  {'-'*78}")

    t_exp_start = time.time()
    yr_prev = yref_v[0]
    x_hat   = x_ss_init.copy()   # inicializar para usar en primer paso

    for k in range(Nsim):
        t_step_start = time.time()
        yr = yref_v[k]

        if yr != yr_prev and yr in SS_NOM:
            x_ss_new, u_ss_new = SS_NOM[yr]
            Xw = np.tile(x_ss_new, (N_mpc+1, 1)).T
            Uw = np.tile(u_ss_new, (N_mpc, 1)).T
            w0_mpc = np.concatenate([Xw.flatten(order='F'), Uw.flatten(order='F')])
            print(f"\n  ▶ Cambio referencia: {yr_prev*100:.0f}→{yr*100:.0f} cm")
        yr_prev = yr

        # ── 1. Leer medición del ESP32 — ahora tres canales ─────────────────
        h1_meas, h2_real, h3_meas = esp.get_measurement(timeout=Ts*2)
        H_meas[0, k+1]  = h1_meas
        H_meas[1, k+1]  = h3_meas
        H2_REAL[k+1]    = h2_real   # solo almacenamiento
        y_k = np.array([h1_meas, h3_meas])   # MHE solo usa h1, h3

        # ── 2. MHE ──────────────────────────────────────────────────────────
        Uk_buf = np.roll(Uk_buf, -1, axis=1); Uk_buf[:, -1] = uprev
        Yk_buf = np.roll(Yk_buf, -1, axis=1); Yk_buf[:, -1] = y_k

        par_mhe_val = np.concatenate([xbar,
                                       Uk_buf.flatten(order='F'),
                                       Yk_buf.flatten(order='F')])
        t_mhe_start = time.time()
        try:
            sol_mhe = solver_mhe(x0=w0_mhe, lbx=lbw_mhe, ubx=ubw_mhe,
                                  p=par_mhe_val)
            X_opt   = np.array(sol_mhe['x']).flatten().reshape(N_mhe+1, nx).T
            x_hat   = np.clip(X_opt[:, -1], 0, 0.22)
            xbar    = np.clip(X_opt[:, 1],  0, 0.22)
            w0_mhe  = np.concatenate([X_opt[:, 1:].flatten(order='F'),
                                       X_opt[:, -1]])
        except Exception as e:
            print(f"  ⚠ MHE falló en k={k}: {e} — usando xbar")
            x_hat = xbar.copy()
        t_mhe_elapsed = (time.time() - t_mhe_start) * 1000
        t_mhe_ms.append(t_mhe_elapsed)
        X_est[:, k+1] = x_hat

        # ── 3. NMPC ─────────────────────────────────────────────────────────
        par_mpc_val = np.concatenate([x_hat, [yr], uprev])
        t_mpc_start = time.time()
        try:
            sol_mpc = solver_mpc(x0=w0_mpc, lbx=lbw_mpc, ubx=ubw_mpc,
                                  lbg=lbg, ubg=ubg, p=par_mpc_val)
            w_opt   = np.array(sol_mpc['x']).flatten()
            Xtraj   = w_opt[:nx*(N_mpc+1)].reshape(N_mpc+1, nx).T
            Utraj   = w_opt[nx*(N_mpc+1):].reshape(N_mpc, nu).T
            u_k     = np.clip(Utraj[:, 0], umin, umax)
            Xsh = np.hstack([Xtraj[:, 1:], Xtraj[:, -1:]])
            Ush = np.hstack([Utraj[:, 1:], Utraj[:, -1:]])
            w0_mpc  = np.concatenate([Xsh.flatten(order='F'), Ush.flatten(order='F')])
        except Exception as e:
            print(f"  ⚠ NMPC falló en k={k}: {e} — usando u anterior")
            u_k = uprev.copy(); w0_mpc = None
        t_mpc_elapsed = (time.time() - t_mpc_start) * 1000
        t_mpc_ms.append(t_mpc_elapsed)

        t_total = (time.time() - t_step_start) * 1000
        t_total_ms.append(t_total)

        U_hist[:, k] = u_k; uprev = u_k.copy()
        esp.send_control(u_k[0], u_k[1])

        if k % 5 == 0:
            e_obs = (x_hat[1] - h2_real) * 100
            print(f"  {k*Ts:>6.0f} | {h1_meas*100:>7.2f} | {x_hat[1]*100:>7.2f} | "
                  f"{h2_real*100:>7.2f} | {e_obs:>+8.3f} | {yr*100:>7.2f} | "
                  f"{u_k[0]:>5.3f} | {t_mhe_elapsed:>5.0f}ms | {t_mpc_elapsed:>5.0f}ms")

        dt_used = time.time() - t_step_start
        if dt_used < Ts * 0.85:
            time.sleep(Ts - dt_used)

    print(f"\n  Experimento finalizado — {(time.time()-t_exp_start)/60:.1f} min")
    print(f"  Outliers rechazados: {esp.outlier_count}")

    return {
        'T_hist'            : T_hist,
        'H_meas'            : H_meas,
        'H2_REAL'           : H2_REAL,   # ← NUEVO
        'X_est'             : X_est,
        'U_hist'            : U_hist,
        'yref_v'            : yref_v,
        't_mhe_ms'          : t_mhe_ms,
        't_mpc_ms'          : t_mpc_ms,
        't_total_ms'        : t_total_ms,
        'ref_change_times_s': ref_change_times_s,
        'outliers'          : esp.outlier_count,
    }

# ============================================================================
# MÉTRICAS
# ============================================================================
def compute_metrics(res):
    T    = res['T_hist']
    ref  = np.append(res['yref_v'], res['yref_v'][-1])
    rct  = res['ref_change_times_s']

    mask_400 = T > 400
    settling_mask = np.ones(len(T), dtype=bool)
    for tc in rct:
        settling_mask &= ~((T >= tc) & (T < tc + SETTLING_S))
    mask_ss = mask_400 & settling_mask

    # Control
    e_ctrl    = (res['X_est'][1, :] - ref) * 100
    rmse_ss   = float(np.sqrt(np.mean(e_ctrl[mask_ss]**2)))
    bias_ctrl = float(np.mean(e_ctrl[mask_ss]))
    mae_ctrl  = float(np.mean(np.abs(e_ctrl[mask_ss])))
    rmse_trans = float(np.sqrt(np.mean(e_ctrl[mask_400]**2)))

    # Observador MHE vs. verdad-terreno
    e_obs        = (res['X_est'][1, :] - res['H2_REAL']) * 100
    rmse_obs_ss  = float(np.sqrt(np.mean(e_obs[mask_ss]**2)))
    bias_obs_ss  = float(np.mean(e_obs[mask_ss]))
    mae_obs_ss   = float(np.mean(np.abs(e_obs[mask_ss])))

    avg_mhe = np.mean(res['t_mhe_ms'])
    avg_mpc = np.mean(res['t_mpc_ms'])
    avg_tot = np.mean(res['t_total_ms'])
    p99_tot = np.percentile(res['t_total_ms'], 99)

    return {
        'rmse_ss'    : rmse_ss,
        'bias_ctrl'  : bias_ctrl,
        'mae_ctrl'   : mae_ctrl,
        'rmse_trans' : rmse_trans,
        'e_ctrl'     : e_ctrl,
        'mask_ss'    : mask_ss,
        'e_obs'      : e_obs,
        'rmse_obs_ss': rmse_obs_ss,
        'bias_obs_ss': bias_obs_ss,
        'mae_obs_ss' : mae_obs_ss,
        'avg_mhe'    : avg_mhe,
        'avg_mpc'    : avg_mpc,
        'avg_tot'    : avg_tot,
        'p99_tot'    : p99_tot,
    }

# ============================================================================
# GRÁFICAS
# ============================================================================
def plot_results(res, save_dir=output_dir):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']
    m   = compute_metrics(res)

    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    fig.suptitle(
        f'MHE+NMPC — Planta Real con sensor verdad-terreno h₂ | '
        f'{datetime.now().strftime("%Y-%m-%d %H:%M")}\n'
        f'RMSE_ctrl={m["rmse_ss"]:.3f}cm | Bias_ctrl={m["bias_ctrl"]:.3f}cm | '
        f'RMSE_obs={m["rmse_obs_ss"]:.3f}cm | Bias_obs={m["bias_obs_ss"]:.3f}cm | '
        f'Total={m["avg_tot"]:.0f}ms/paso',
        fontsize=10, fontweight='bold')

    # ── Fila 1: h1 y h3 ─────────────────────────────────────────────────────
    ax = axes[0, 0]
    ax.plot(T, res['H_meas'][0]*100, 'b', lw=1.5, label='$h_1$ medido')
    ax.plot(T, res['X_est'][0]*100,  'r--', lw=1.5, label='$h_1$ estimado (MHE)')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 1 (medido)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(T, res['H_meas'][1]*100, 'b', lw=1.5, label='$h_3$ medido')
    ax.plot(T, res['X_est'][2]*100,  'r--', lw=1.5, label='$h_3$ estimado (MHE)')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 3 (medido)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Fila 2: h2 estimado vs. real vs. referencia ──────────────────────────
    ax = axes[1, 0]
    ax.plot(T, res['H2_REAL']*100,  'g',   lw=1.8,
            label='$h_2$ real (sensor verificación)')
    ax.plot(T, res['X_est'][1]*100, 'r--', lw=2.0,
            label=f'$h_2$ estimado MHE (RMSE_obs={m["rmse_obs_ss"]:.3f}cm)')
    ax.stairs(ref[:-1]*100, T, color='k', linestyle='--', lw=1.5, label='Referencia')
    for tc in rct:
        ax.axvline(tc, color='gray', lw=0.8, ls=':')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 2 — estimado MHE vs. sensor verdad-terreno')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Fila 2 derecha: error puro del observador MHE ────────────────────────
    ax = axes[1, 1]
    ax.plot(T, m['e_obs'], color='darkgreen', lw=1.5,
            label=(f'$e_{{obs}}$ = $\\hat{{h}}_2$ − $h_2^{{real}}$ | '
                   f'RMSE={m["rmse_obs_ss"]:.3f}cm | '
                   f'Bias={m["bias_obs_ss"]:.3f}cm'))
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error observador [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error puro del observador MHE (estimado − real)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Fila 3: error de control y señales de control ────────────────────────
    ax = axes[2, 0]
    ax.plot(T, m['e_ctrl'], 'r', lw=1.5,
            label=(f'RMSE_SS={m["rmse_ss"]:.3f}cm | '
                   f'RMSE_trans={m["rmse_trans"]:.3f}cm | '
                   f'Bias={m["bias_ctrl"]:.3f}cm'))
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error ctrl $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error de seguimiento'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.stairs(res['U_hist'][0], T, color='b', lw=1.5, label='$u_1$')
    ax.stairs(res['U_hist'][1], T, color='r', lw=1.5, label='$u_3$')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]')
    ax.set_xlabel('t [s]'); ax.set_title('Entradas NMPC')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Fila 4: tiempos y tabla ───────────────────────────────────────────────
    ax = axes[3, 0]
    ks = np.arange(len(res['t_total_ms']))
    ax.plot(ks, res['t_mhe_ms'],   'b', lw=0.8, alpha=0.7,
            label=f'MHE ({m["avg_mhe"]:.0f}ms)')
    ax.plot(ks, res['t_mpc_ms'],   'g', lw=0.8, alpha=0.7,
            label=f'NMPC ({m["avg_mpc"]:.0f}ms)')
    ax.plot(ks, res['t_total_ms'], 'k', lw=1.2,
            label=f'Total ({m["avg_tot"]:.0f}ms)')
    ax.axhline(Ts*1000, color='r', lw=1.5, ls='--', label=f'Ts={Ts*1000:.0f}ms')
    ax.set_ylabel('Tiempo [ms]'); ax.set_xlabel('Paso k')
    ax.set_title('Tiempos de cómputo por paso'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[3, 1]
    ax.axis('off')
    tabla = [
        ['Métrica', 'MHE+NMPC', 'Koopman v4b'],
        ['RMSE ctrl h₂ SS [cm]',    f'{m["rmse_ss"]:.3f}',    '0.281'],
        ['MAE  ctrl h₂ SS [cm]',    f'{m["mae_ctrl"]:.3f}',   '0.234'],
        ['Bias ctrl h₂ SS [cm]',    f'{m["bias_ctrl"]:.3f}',  '0.006'],
        ['─────────────────────', '──────────', '──────────'],
        ['RMSE obs h₂ SS [cm]',     f'{m["rmse_obs_ss"]:.3f}','0.557'],
        ['Bias obs h₂ SS [cm]',     f'{m["bias_obs_ss"]:.3f}','−0.291'],
        ['MAE  obs h₂ SS [cm]',     f'{m["mae_obs_ss"]:.3f}', '0.444'],
        ['─────────────────────', '──────────', '──────────'],
        ['MHE media [ms]',           f'{m["avg_mhe"]:.1f}',   '—'],
        ['NMPC media [ms]',          f'{m["avg_mpc"]:.1f}',   '—'],
        ['Total medio [ms]',         f'{m["avg_tot"]:.1f}',   '2099'],
        ['Total p99  [ms]',          f'{m["p99_tot"]:.1f}',   '2125'],
    ]
    t = ax.table(cellText=tabla[1:], colLabels=tabla[0],
                 loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9)
    t.scale(1.1, 1.4)
    ax.set_title('Comparación de esquemas', fontsize=10, fontweight='bold', pad=10)

    plt.tight_layout()
    tstamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname  = save_dir / f'mhe_nmpc_real_{tstamp}.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
    print(f"\n  Figura guardada: {fname}")

    # CSV con columna h2_real y error_obs
    csv_path = save_dir / f'mhe_nmpc_real_{tstamp}.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['t_s', 'h1_meas_m', 'h2_est_m', 'h2_real_m', 'error_obs_cm',
                    'h3_meas_m', 'ref_m', 'u1', 'u3',
                    't_mhe_ms', 't_mpc_ms', 't_total_ms', 'error_ctrl_cm'])
        for k in range(Nsim):
            e_ctrl = (res['X_est'][1, k] - res['yref_v'][k]) * 100
            e_obs  = (res['X_est'][1, k] - res['H2_REAL'][k]) * 100
            w.writerow([
                T[k],
                res['H_meas'][0, k],  res['X_est'][1, k],
                res['H2_REAL'][k],    e_obs,
                res['H_meas'][1, k],
                res['yref_v'][k],
                res['U_hist'][0, k],  res['U_hist'][1, k],
                res['t_mhe_ms'][k],   res['t_mpc_ms'][k],  res['t_total_ms'][k],
                e_ctrl,
            ])
    print(f"  Datos guardados: {csv_path}")

# ============================================================================
# RESUMEN EN CONSOLA
# ============================================================================
def print_summary(res):
    m = compute_metrics(res)
    print(f"\n  {'='*60}")
    print(f"  RESUMEN MHE+NMPC — PLANTA REAL (con verdad-terreno h₂)")
    print(f"  {'='*60}")
    print(f"\n  MÉTRICAS DE CONTROL (ĥ₂_MHE vs. referencia):")
    print(f"  {'RMSE seguimiento h₂ (SS):':<40} {m['rmse_ss']:.4f} cm")
    print(f"  {'RMSE seguimiento h₂ (transitorio):':<40} {m['rmse_trans']:.4f} cm")
    print(f"  {'MAE  seguimiento h₂ (SS):':<40} {m['mae_ctrl']:.4f} cm")
    print(f"  {'Bias seguimiento h₂ (SS):':<40} {m['bias_ctrl']:.4f} cm")
    print(f"\n  MÉTRICAS DEL OBSERVADOR MHE (ĥ₂ vs. h₂ real — sensor verificación):")
    print(f"  {'RMSE estimación h₂ (SS):':<40} {m['rmse_obs_ss']:.4f} cm")
    print(f"  {'Bias estimación h₂ (SS):':<40} {m['bias_obs_ss']:.4f} cm")
    print(f"  {'MAE  estimación h₂ (SS):':<40} {m['mae_obs_ss']:.4f} cm")
    print(f"\n  TIEMPOS DE CÓMPUTO:")
    print(f"  {'MHE  — media:':<40} {m['avg_mhe']:.1f} ms/paso")
    print(f"  {'NMPC — media:':<40} {m['avg_mpc']:.1f} ms/paso")
    print(f"  {'Total — media:':<40} {m['avg_tot']:.1f} ms | p99: {m['p99_tot']:.1f} ms")
    print(f"  {'Outliers rechazados:':<40} {res['outliers']}")
    print(f"\n  TABLA COMPARATIVA (para tesis):")
    print(f"  {'Métrica':<38} {'MHE+NMPC':>10} {'Koopman v4b':>12}")
    print(f"  {'-'*62}")
    print(f"  {'RMSE ctrl h₂ SS [cm]':<38} {m['rmse_ss']:>10.3f} {'0.281':>12}")
    print(f"  {'MAE  ctrl h₂ SS [cm]':<38} {m['mae_ctrl']:>10.3f} {'0.234':>12}")
    print(f"  {'Bias ctrl h₂ SS [cm]':<38} {m['bias_ctrl']:>10.3f} {'0.006':>12}")
    print(f"  {'RMSE obs  h₂ SS [cm]':<38} {m['rmse_obs_ss']:>10.3f} {'0.557':>12}")
    print(f"  {'Bias obs  h₂ SS [cm]':<38} {m['bias_obs_ss']:>10.3f} {'−0.291':>12}")
    print(f"  {'Tiempo medio [ms]':<38} {m['avg_tot']:>10.1f} {'2099':>12}")
    print(f"  {'='*60}")

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 65)
    print("  MHE + NMPC — PLANTA FÍSICA REAL (con verdad-terreno h₂)")
    print(f"  Puerto: {SERIAL_PORT} | Ts={Ts}s | T_sim={T_sim}s | {Nsim} pasos")
    print("=" * 65)

    print("\nCalculando puntos de operación...")
    SS_NOM = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS_NOM[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)} | u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → ✗ No convergió")
    if not SS_NOM:
        print("  ERROR: Sin puntos estacionarios.")
        sys.exit(1)

    solvers = build_solvers(P_NOM)

    esp = ESP32Interface(port=SERIAL_PORT, baud=BAUD_RATE)
    try:
        esp.connect()

        print("\n  Verificación de sensores (5 lecturas) — incluye h₂:")
        for i in range(5):
            h1, h2r, h3 = esp.get_measurement()
            print(f"    [{i+1}] h1={h1*100:.2f}cm | h2_real={h2r*100:.2f}cm | h3={h3*100:.2f}cm")
            time.sleep(0.5)

        input("\n  ¿Todo OK? Presiona ENTER para iniciar (Ctrl+C para cancelar)...")

        res = run_real_experiment(esp, SS_NOM, solvers)
        print_summary(res)
        plot_results(res)

    except KeyboardInterrupt:
        print("\n\n  Interrumpido por el usuario.")
    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback; traceback.print_exc()
    finally:
        esp.close()

if __name__ == '__main__':
    main()

  MHE + NMPC — PLANTA FÍSICA REAL (con verdad-terreno h₂)
  Puerto: COM7 | Ts=2.0s | T_sim=800s | 400 pasos

Calculando puntos de operación...
  h2=0.10m → x_ss=[0.114  0.1    0.0816] | u_ss=[0.3293 0.    ] ✓
  h2=0.15m → x_ss=[0.1677 0.15   0.1267] | u_ss=[0.4046 0.    ] ✓
  h2=0.20m → ✗ No convergió

Construyendo modelo CasADi...
  Modelo RK4 listo.
  Construyendo MHE...
  MHE listo (N=20).
  Construyendo NMPC...
  NMPC listo (N=20, q=500).

Conectando a COM7 @ 115200 baud...
  ✓ Conectado a COM7

  Verificación de sensores (5 lecturas) — incluye h₂:
    [1] h1=1.91cm | h2_real=1.61cm | h3=1.11cm
    [2] h1=1.91cm | h2_real=1.59cm | h3=1.11cm
    [3] h1=1.91cm | h2_real=1.59cm | h3=1.13cm
    [4] h1=1.91cm | h2_real=1.59cm | h3=1.11cm
    [5] h1=1.91cm | h2_real=1.59cm | h3=1.11cm

Esperando primera medición del ESP32...
  Primera medición: h1=1.91cm | h2_real=1.59cm | h3=1.13cm

  MHE+NMPC PLANTA REAL (con sensor verdad-terreno h₂) — 10:55:33
  h₂_real: solo verificación — NO entra 